# ETA Error Time Series — Aleph RF–inspired

**Project context**
- **Dataset:** `data_etas.csv` exported from the Aleph Random Forest ETA notebook. Each row is a stop-to-stop segment (origin → next stop on same route).
- **Timestamps:** `real_departure_origin`, `real_arrival_dest`, `eta_arrival_dest`. We compute real vs estimated trip time in minutes and filter valid rows (both > 0, non-null). ETA error = real_trip_time_minutes − estimated_trip_time_minutes.
- **Segment ID:** `trip_id` = origin–destination pair (repeats across days/routes). We pick one segment and turn irregular observations into an evenly spaced series for forecasting.
- **Regularization:** 6-hour buckets; aggregate with median; fill missing buckets with weekday × 6h-slot median baseline, then global median fallback. Final format: `unique_id`, `ds`, `y`. `ds` is kept timezone-naive for Prophet/TimeCopilot.

## 1. Load
- Read `data_etas.csv` (stop-to-stop segments from Aleph RF ETA notebook).
- Simple error handling if file is missing.

In [1]:
import os
import pandas as pd
import numpy as np
import nest_asyncio
from timecopilot import TimeCopilot
from timecopilot import TimeCopilotForecaster
from timecopilot.models.stats import SeasonalNaive, Theta
from timecopilot.models.foundation.chronos import Chronos

from timecopilot.models.prophet import Prophet
from timecopilot.models.stats import AutoARIMA, AutoETS, SeasonalNaive
from timecopilot.models.foundation.moirai import Moirai
from timecopilot.models.foundation.timesfm import TimesFM
from timecopilot.models.foundation.tirex import TiRex

api_key = os.environ["OPENAI_API_KEY"]

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


In [ ]:
def import_data(file_name):
    """Load segment-level ETA data from CSV. Raises clear error if file not found."""
    if not os.path.isfile(file_name):
        raise FileNotFoundError(f"Data file not found: {file_name}")
    return pd.read_csv(file_name)

data_trips = import_data("data_etas.csv")

## 2. Parse / Compute
- Parse timestamp columns; compute real and estimated trip times; convert to minutes.
- Same logic as the Aleph RF ETA notebook.

In [ ]:
time_cols = [
    "real_departure_origin",
    "real_arrival_dest",
    "eta_arrival_dest",
]
for col in time_cols:
    data_trips[col] = pd.to_datetime(data_trips[col], errors="coerce")

data_trips["real_trip_time"] = (
    data_trips["real_arrival_dest"] - data_trips["real_departure_origin"]
)
data_trips["estimated_trip_time"] = (
    data_trips["eta_arrival_dest"] - data_trips["real_departure_origin"]
)

data_trips["real_trip_time_minutes"] = (
    data_trips["real_trip_time"].dt.total_seconds() / 60
)
data_trips["estimated_trip_time_minutes"] = (
    data_trips["estimated_trip_time"].dt.total_seconds() / 60
)


## 3. Filter
- Keep rows where both real and estimated trip time (minutes) are non-null and > 0.
- Compute ETA error in minutes: real_trip_time_minutes − estimated_trip_time_minutes.

There's a slight change in this version. eta error is handled as a multiple error. Going back to
the additive approach is straightforward. Just change the division for subtraction and uncomment 
the log normalization (outliers aren't as terrible in this form)

In [ ]:
valid_mask = (
    data_trips["real_trip_time_minutes"].notna()
    & (data_trips["real_trip_time_minutes"] > 0)
    & data_trips["estimated_trip_time_minutes"].notna()
    & (data_trips["estimated_trip_time_minutes"] > 0)
)
data_valid_trips = data_trips.loc[valid_mask].copy()

data_valid_trips["eta_error_minutes"] = (
    data_valid_trips["real_trip_time_minutes"]
    / data_valid_trips["estimated_trip_time_minutes"]
)

print("Total rows:", len(data_trips))
print("Valid segments:", len(data_valid_trips))
data_valid_trips.head()

Total rows: 586164
Valid segments: 345664


,external_id,lat_origin,lon_origin,location_id_origin,real_departure_origin,external_schedule_departure_origin,real_seq,lat_dest,lon_dest,location_id_dest,real_arrival_dest,external_schedule_arrival_dest,eta_arrival_dest,real_trip_time,estimated_trip_time,real_trip_time_minutes,estimated_trip_time_minutes,trip_id,eta_error_minutes
16,ae9503ad-81ad-417b-a49b-6cc67ec3d7d5,25.707150,-100.279370,9820891a-be20-4ebd-a402-655ec6c6844c,2025-07-22 02:41:39+00:00,2025-07-21 18:48:27+00:00,1,19.461210,-99.152280,2463c639-c724-4caf-b922-13fa0b270442,2025-07-23 00:42:47+00:00,2025-07-22 14:45:00+00:00,2025-07-22 18:07:36+00:00,0 days 22:01:08,0 days 15:25:57,1321.133333,925.950000,9820891a-be20-4ebd-a402-655ec6c6844c-2463c639-...,1.426787
17,ae9503ad-81ad-417b-a49b-6cc67ec3d7d5,19.461210,-99.152280,2463c639-c724-4caf-b922-13fa0b270442,2025-07-23 00:43:00+00:00,2025-07-22 16:45:00+00:00,2,25.707150,-100.279370,9820891a-be20-4ebd-a402-655ec6c6844c,2025-07-26 05:50:31+00:00,2025-07-23 11:49:57+00:00,2025-07-26 05:50:53+00:00,3 days 05:07:31,3 days 05:07:53,4627.516667,4627.883333,2463c639-c724-4caf-b922-13fa0b270442-9820891a-...,0.999921
18,fd8798f9-fa99-4e33-b726-633e8e39dd23,19.658553,-99.170608,cf2dbff4-1216-4843-9ccb-a87427a1e98a,2025-01-04 01:29:07+00:00,2025-01-04 04:00:00+00:00,1,25.624188,-103.510615,67c0d710-d3d0-49c5-af2f-4e191d16186f,2025-01-05 18:55:46+00:00,2025-01-05 01:26:12+00:00,2025-01-05 19:03:53+00:00,1 days 17:26:39,1 days 17:34:46,2486.650000,2494.766667,cf2dbff4-1216-4843-9ccb-a87427a1e98a-67c0d710-...,0.996747
19,fd8798f9-fa99-4e33-b726-633e8e39dd23,25.624188,-103.510615,67c0d710-d3d0-49c5-af2f-4e191d16186f,2025-01-06 15:11:19+00:00,2025-01-05 02:56:12+00:00,2,28.719916,-106.139718,1cc50ee6-df0e-4484-8614-b1cd341f1db8,2025-01-06 22:42:49+00:00,2025-01-05 10:54:25+00:00,2025-01-06 22:57:05+00:00,0 days 07:31:30,0 days 07:45:46,451.500000,465.766667,67c0d710-d3d0-49c5-af2f-4e191d16186f-1cc50ee6-...,0.969369
20,fd8798f9-fa99-4e33-b726-633e8e39dd23,28.719916,-106.139718,1cc50ee6-df0e-4484-8614-b1cd341f1db8,2025-01-08 15:28:40+00:00,2025-01-05 12:24:25+00:00,3,19.658553,-99.170608,cf2dbff4-1216-4843-9ccb-a87427a1e98a,2025-01-10 01:38:28+00:00,2025-01-06 21:42:08+00:00,2025-01-10 01:50:22+00:00,1 days 10:09:48,1 days 10:21:42,2049.800000,2061.700000,1cc50ee6-df0e-4484-8614-b1cd341f1db8-cf2dbff4-...,0.994228


## 4. Segment selection
- Pick one segment (`trip_id`) to turn into a regular time series for forecasting.
- We will select `TARGET_TRIP_ID` as a list of target id routes. We will try the 10 routes with more data. Trying to take the ones with few samples isn't great for the moment because there's several with just one registry. 

Notice there's a total of 3633 routes, so plenty to choose from.

In [10]:
trip_kounts = pd.DataFrame(data_valid_trips.drop_duplicates(subset=['trip_id','real_departure_origin'])['trip_id'].value_counts()).reset_index(drop=False)
trip_kounts.to_csv('trip_kounts.csv',index=False)

In [5]:
# Count trips
trip_counts = data_valid_trips['trip_id'].value_counts()
print(trip_counts)

# Top 10 most frequent
# Changed to 50 but not did much to change variable names. 
trips2run=50
top_10_ids = trip_counts.head(trips2run).index.tolist()

# # Bottom 5 least frequent
# bottom_5_ids = trip_counts.tail(5).index.tolist()
# Tried going for the least count but lots have only one, so not ideal. 

# Combine
selected_ids = top_10_ids # + bottom_5_ids

# Filter dataframe
segment_df = data_valid_trips[data_valid_trips['trip_id'].isin(selected_ids)]

# Print for nefarious purposes
print("Top trip_ids by count:")
print(top_10_ids)

# print("\nBottom 5 trip_ids by count:")
# print(bottom_5_ids)

# TARGET_TRIP_ID = (
#     "71a24abd-0fb8-4110-be27-a93d8078d2cf-622564d0-7320-40f0-bb72-3ead8356e87c"
# )
# segment_df = data_valid_trips[
#     data_valid_trips["trip_id"] == TARGET_TRIP_ID
# ].copy()

trip_id
3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0-8deed86c-92aa-4a07-aaf2-095a6affc7d0    7373
8deed86c-92aa-4a07-aaf2-095a6affc7d0-3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0    4640
2463c639-c724-4caf-b922-13fa0b270442-71a24abd-0fb8-4110-be27-a93d8078d2cf    3623
71a24abd-0fb8-4110-be27-a93d8078d2cf-71a24abd-0fb8-4110-be27-a93d8078d2cf    2932
f0c17036-5c77-496c-9611-7844c363cb44-85b12c06-c079-41d9-812a-3ee1c0ea37bd    2624
                                                                             ... 
093fde72-23c8-43d7-b24a-31ed1ed4e134-203795bb-ac58-447b-a708-9ce62b202d4c       1
714161ad-dffc-4fb6-80d1-9d6add017ee8-e629b935-5675-4907-824e-16ed9fc11bf0       1
ac979b5b-a36a-4ec1-98d6-552f781934dc-e1ea991a-17da-4fd4-82fe-f8628b9fe819       1
6d107199-8f80-4682-a744-968156c1955c-192646c1-bd87-4509-a82b-b5f64fc20a28       1
1b8cab9a-d4aa-4acf-b5c1-f20f7398703b-a471abde-d9fc-4f1d-9dc0-3207f7247d97       1
Name: count, Length: 3633, dtype: int64
Top trip_ids by count:
['3a291b8e-7a0a-4fc8-8deb-1

## 5. Regularize to 6H buckets & imputation
- Build an evenly spaced 6H series from irregular segment observations: signed-log ETA error, then resample with median (or mean).
- Fill missing buckets: weekday × 6h-slot median baseline; fallback to global median. Output: `unique_id`, `ds`, `y`.

In [6]:
def build_6h_even_series(data_valid_trips, trip_id, agg="median"):
    """
    Build an evenly spaced 6H time series for one segment.

    Inputs:
      data_valid_trips: DataFrame with columns trip_id, real_departure_origin, eta_error_minutes.
      trip_id: Segment identifier (origin-destination pair).
      agg: Aggregation for 6H buckets — "median" (default) or "mean".

    Output:
      (final_df, impute_info): final_df has columns unique_id, ds, y; impute_info is
      {"n_imputed": int, "n_total": int} for sanity checks.

    Logic: Filter segment → signed-log(eta_error_minutes) → resample 6H with agg →
    fill missing buckets with weekday×slot6h median baseline, then global median fallback.
    """
    required = ["trip_id", "real_departure_origin", "eta_error_minutes"]
    for col in required:
        if col not in data_valid_trips.columns:
            raise ValueError(f"Required column missing: {col}")

    df = data_valid_trips[data_valid_trips["trip_id"] == trip_id].copy()
    if df.empty:
        raise ValueError(f"No rows found for trip_id: {trip_id!r}")

    df["real_departure_origin"] = pd.to_datetime(
        df["real_departure_origin"], utc=True, errors="coerce"
    )
    df = (
        df.dropna(subset=["real_departure_origin"])
        .sort_values("real_departure_origin")
        .set_index("real_departure_origin")
    )

    e = df["eta_error_minutes"].astype(float)
    df["signed_log_error"] = np.sign(e) * np.log1p(np.abs(e))

    if agg == "median": 
        y = df["signed_log_error"].resample("6H").median()
    elif agg == "mean":
        y = df["signed_log_error"].resample("6H").mean()
    else:
        raise ValueError(f"agg must be 'median' or 'mean', got {agg!r}")

    missing = y.isna()
    tmp = pd.DataFrame({"y": y})
    tmp["weekday"] = tmp.index.dayofweek
    tmp["slot6h"] = tmp.index.hour // 6

    slot_median = (
        tmp.loc[~missing].groupby(["weekday", "slot6h"])["y"].median()
    )
    y_filled = y.copy()
    fill_vals = tmp.loc[missing, ["weekday", "slot6h"]].apply(
        lambda r: slot_median.get((r["weekday"], r["slot6h"])), axis=1
    )
    y_filled.loc[missing] = fill_vals.values
    y_filled = y_filled.fillna(y.median())

    final_df = pd.DataFrame({
        "unique_id": trip_id,
        "ds": y_filled.index,
        "y": y_filled.values,
    })
    impute_info = {"n_imputed": int(missing.sum()), "n_total": len(y)}
    return final_df, impute_info

# Created an analogue version of the same function that will take in a list for trip_id 
# in order to process several in one run. 
def build_6h_even_series2(data_valid_trips, trip_ids, agg="median"):
    """
    Build evenly spaced 6H time series for one or multiple trip_ids.

    Inputs:
      data_valid_trips: DataFrame with columns trip_id, real_departure_origin, eta_error_minutes.
      trip_ids: str/int trip_id OR list/iterable of trip_ids.
      agg: Aggregation for 6H buckets — "median" (default) or "mean".

    Output:
      (final_df, impute_info):
        - final_df has columns unique_id, ds, y for ALL requested trip_ids concatenated
        - impute_info is a dict keyed by trip_id with {"n_imputed": int, "n_total": int}
    """
    required = ["trip_id", "real_departure_origin", "eta_error_minutes"]
    for col in required:
        if col not in data_valid_trips.columns:
            raise ValueError(f"Required column missing: {col}")

    # Normalize trip_ids to a list
    if isinstance(trip_ids, (str, int, np.integer)):
        trip_ids = [trip_ids]
    else:
        trip_ids = list(trip_ids)

    if len(trip_ids) == 0:
        raise ValueError("trip_ids is empty.")

    # Parse datetime once (cheaper + consistent)
    df_all = data_valid_trips.copy()
    df_all["real_departure_origin"] = pd.to_datetime(
        df_all["real_departure_origin"], utc=True, errors="coerce"
    )
    df_all = df_all.dropna(subset=["real_departure_origin"])

    # Pre-filter once
    df_all = df_all[df_all["trip_id"].isin(trip_ids)].copy()
    if df_all.empty:
        raise ValueError(f"No rows found for trip_ids: {trip_ids!r}")

    out_frames = []
    impute_info = {}

    for trip_id in trip_ids:
        df = df_all[df_all["trip_id"] == trip_id].copy()
        if df.empty:
            # If you prefer to skip missing trip_ids instead of raising, change this.
            raise ValueError(f"No rows found for trip_id: {trip_id!r}")

        df = (
            df.sort_values("real_departure_origin")
              .set_index("real_departure_origin")
        )

        e = df["eta_error_minutes"].astype(float)

        # This is to be uncommented if working with the additive approach
        df["signed_log_error"] = e # np.sign(e) * np.log1p(np.abs(e))

        if agg == "median":
            y = df["signed_log_error"].resample("6H").median()
        elif agg == "mean":
            y = df["signed_log_error"].resample("6H").mean()
        else:
            raise ValueError(f"agg must be 'median' or 'mean', got {agg!r}")

        missing = y.isna()

        tmp = pd.DataFrame({"y": y})
        tmp["weekday"] = tmp.index.dayofweek
        tmp["slot6h"] = tmp.index.hour // 6

        slot_median = tmp.loc[~missing].groupby(["weekday", "slot6h"])["y"].median()

        y_filled = y.copy()
        fill_vals = tmp.loc[missing, ["weekday", "slot6h"]].apply(
            lambda r: slot_median.get((r["weekday"], r["slot6h"])),
            axis=1,
        )
        y_filled.loc[missing] = fill_vals.values

        # Global fallback
        y_filled = y_filled.fillna(y.median())

        final_df_one = pd.DataFrame({
            "unique_id": trip_id,
            "ds": y_filled.index,
            "y": y_filled.values,
        })

        out_frames.append(final_df_one)
        impute_info[trip_id] = {"n_imputed": int(missing.sum()), "n_total": int(len(y))}

    final_df = pd.concat(out_frames, ignore_index=True)
    return final_df, impute_info

In [ ]:
segment_df_final, impute_info = build_6h_even_series2(
    data_valid_trips, selected_ids
)

,unique_id,ds,y
0,3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0-8deed86c-...,2025-01-03 12:00:00+00:00,0.020461
1,3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0-8deed86c-...,2025-01-03 18:00:00+00:00,0.030685
2,3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0-8deed86c-...,2025-01-04 00:00:00+00:00,0.015053
3,3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0-8deed86c-...,2025-01-04 06:00:00+00:00,0.029370
4,3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0-8deed86c-...,2025-01-04 12:00:00+00:00,0.026408
...,...,...,...
60203,71a24abd-0fb8-4110-be27-a93d8078d2cf-c787140d-...,2025-10-29 00:00:00+00:00,0.794768
60204,71a24abd-0fb8-4110-be27-a93d8078d2cf-c787140d-...,2025-10-29 06:00:00+00:00,0.516798
60205,71a24abd-0fb8-4110-be27-a93d8078d2cf-c787140d-...,2025-10-29 12:00:00+00:00,1.085740
60206,71a24abd-0fb8-4110-be27-a93d8078d2cf-c787140d-...,2025-10-29 18:00:00+00:00,0.774762


## 6. Sanity check
- Shape, date range, and share of buckets that were imputed; head and tail of the series.

In [8]:
n_imp = sum(v["n_imputed"] for v in impute_info.values())
n_tot = sum(v["n_total"] for v in impute_info.values())
pct = (n_imp / n_tot * 100) if n_tot else 0

print("Shape:", segment_df_final.shape)
print("Date range:", segment_df_final["ds"].min(), "->", segment_df_final["ds"].max())
print(f"Buckets imputed: {n_imp} / {n_tot} ({pct:.1f}%)")

print("Trip IDs:", segment_df_final["unique_id"].unique().tolist())


Shape: (60208, 3)
Date range: 2025-01-01 12:00:00+00:00 -> 2025-10-30 18:00:00+00:00
Buckets imputed: 25717 / 60208 (42.7%)
Trip IDs: ['3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0-8deed86c-92aa-4a07-aaf2-095a6affc7d0', '8deed86c-92aa-4a07-aaf2-095a6affc7d0-3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0', '2463c639-c724-4caf-b922-13fa0b270442-71a24abd-0fb8-4110-be27-a93d8078d2cf', '71a24abd-0fb8-4110-be27-a93d8078d2cf-71a24abd-0fb8-4110-be27-a93d8078d2cf', 'f0c17036-5c77-496c-9611-7844c363cb44-85b12c06-c079-41d9-812a-3ee1c0ea37bd', '1ac94e88-c453-45c8-a30f-e21c0e8077c5-622564d0-7320-40f0-bb72-3ead8356e87c', '85b12c06-c079-41d9-812a-3ee1c0ea37bd-f0c17036-5c77-496c-9611-7844c363cb44', 'abff9b86-a503-4fab-951a-05672ce62302-e0de46a7-8e4b-4e95-b93e-d01730dbdb7a', '622564d0-7320-40f0-bb72-3ead8356e87c-622564d0-7320-40f0-bb72-3ead8356e87c', '7077bcbc-011e-48d3-a68b-35660b65f76a-9820891a-be20-4ebd-a402-655ec6c6844c', '9820891a-be20-4ebd-a402-655ec6c6844c-7077bcbc-011e-48d3-a68b-35660b65f76a', 'e629b935-5675-490

## 7. Final dataset (timezone-naive)
Prophet (used by TimeCopilot) does not support timezone-aware `ds`. Strip timezone before forecasting.

In [9]:
segment_df_final = segment_df_final.copy()
segment_df_final["ds"] = pd.to_datetime(segment_df_final["ds"]).dt.tz_localize(None)


In [10]:
segment_df_final

,unique_id,ds,y
0,3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0-8deed86c-...,2025-01-03 12:00:00,0.020461
1,3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0-8deed86c-...,2025-01-03 18:00:00,0.030685
2,3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0-8deed86c-...,2025-01-04 00:00:00,0.015053
3,3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0-8deed86c-...,2025-01-04 06:00:00,0.029370
4,3a291b8e-7a0a-4fc8-8deb-12256a2d1ba0-8deed86c-...,2025-01-04 12:00:00,0.026408
...,...,...,...
60203,71a24abd-0fb8-4110-be27-a93d8078d2cf-c787140d-...,2025-10-29 00:00:00,0.794768
60204,71a24abd-0fb8-4110-be27-a93d8078d2cf-c787140d-...,2025-10-29 06:00:00,0.516798
60205,71a24abd-0fb8-4110-be27-a93d8078d2cf-c787140d-...,2025-10-29 12:00:00,1.085740
60206,71a24abd-0fb8-4110-be27-a93d8078d2cf-c787140d-...,2025-10-29 18:00:00,0.774762


Let's find out how much data we want to actually forecast. 

In [11]:
cand = segment_df_final.unique_id.iloc[-1]
print(cand)
print(len(segment_df_final[segment_df_final.unique_id==cand]))

71a24abd-0fb8-4110-be27-a93d8078d2cf-c787140d-e71c-4002-ba21-ea327d028547
1206


Let's try to forecast 100 datapoints. There's enough info, even though it may feel like a stretch. 

In [12]:
segment_df_final = segment_df_final.sort_values(["unique_id", "ds"])

# last 100 per unique_id → holdout
howmany = 100
k = 0 # k is a dropper value that I used to work around the whole api saturation. 
# but also, not necessary anymore due to change of the function calling and manually 
# building an ensemble

holdout_df = (
    segment_df_final.groupby("unique_id", group_keys=False)
      .tail(howmany)
)

# everything else → curtailed / training df
curtailed_df = (
    segment_df_final
      .groupby("unique_id", group_keys=False)
      .apply(lambda x: x.iloc[k:-howmany] if howmany > 0 else x.iloc[k:])
      .reset_index(drop=True)
)

print("Original:", segment_df_final.shape)
print("Curtailed:", curtailed_df.shape)
print("Holdout:", holdout_df.shape)

# per-id counts
# print("\nHoldout counts per id:")
# print(holdout_df["unique_id"].value_counts())

display(curtailed_df)


Original: (60208, 3)
Curtailed: (55208, 3)
Holdout: (5000, 3)


,unique_id,ds,y
0,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-01-02 18:00:00,1.007099
1,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-01-03 00:00:00,1.024460
2,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-01-03 06:00:00,1.003623
3,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-01-03 12:00:00,1.118318
4,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-01-03 18:00:00,1.003017
...,...,...,...
55203,fc7ef67f-5553-4692-aebe-16da7cfe224b-b7d32e46-...,2025-10-04 06:00:00,0.778287
55204,fc7ef67f-5553-4692-aebe-16da7cfe224b-b7d32e46-...,2025-10-04 12:00:00,1.000490
55205,fc7ef67f-5553-4692-aebe-16da7cfe224b-b7d32e46-...,2025-10-04 18:00:00,1.287514
55206,fc7ef67f-5553-4692-aebe-16da7cfe224b-b7d32e46-...,2025-10-05 00:00:00,0.508271


## 8. TimeCopilot forecast
- Use TimeCopilot to select a model and produce a 6H-ahead forecast (e.g. `h=20` buckets).
- Set `OPENAI_API_KEY` via environment variable; do not hardcode secrets in the notebook.

In [13]:
nest_asyncio.apply()

OPENAI API doesn't like big names. replacing for tokenization. 

In [14]:
uid2short = {u: str(i+1) for i, u in enumerate(sorted(curtailed_df["unique_id"].unique()))}
df_tok = curtailed_df.replace({"unique_id": uid2short}).rename(columns={"unique_id":"1","ds":"2","y":"3"})
df_tok

,1,2,3
0,1,2025-01-02 18:00:00,1.007099
1,1,2025-01-03 00:00:00,1.024460
2,1,2025-01-03 06:00:00,1.003623
3,1,2025-01-03 12:00:00,1.118318
4,1,2025-01-03 18:00:00,1.003017
...,...,...,...
55203,50,2025-10-04 06:00:00,0.778287
55204,50,2025-10-04 12:00:00,1.000490
55205,50,2025-10-04 18:00:00,1.287514
55206,50,2025-10-05 00:00:00,0.508271


In [15]:
os.environ["OPENAI_API_KEY"] = api_key
batch_size = 64

I changed the models to be considered within the process of forecasting. 
Also, changed the function calling in order to avoid the high tokenization count. 

Model choice will be approached by manually creating the ensemble. 

In [ ]:
# Setup TimeCopilotForecaster object and load the models that will operate on it
tcf = TimeCopilotForecaster(
    models=[
        AutoARIMA(), 
        Chronos(repo_id="amazon/chronos-bolt-mini"),
        TimesFM(
            repo_id="google/timesfm-2.5-200m-pytorch",
            batch_size=batch_size,  
        ),
        TiRex(
            batch_size=batch_size,
                ),
        Theta(),
        Moirai(), 
        SeasonalNaive(),
        ]
)

result_df = tcf.forecast(df=curtailed_df, freq='6H',h=howmany)
# og=result_df.copy()


100%|██████████| 1/1 [02:05<00:00, 125.16s/it]
50it [01:13,  1.46s/it]


In [17]:
short2uid = {v: k for k, v in uid2short.items()}
result_df = result_df.rename(columns={"1":"unique_id","2":"ds","3":"y"}).replace({"unique_id": short2uid})

In [ ]:
forecast_cols = result_df.columns.difference(["unique_id", "ds"])

# Creating several ensembles for metric comparison
result_df["meanensemble"] = result_df[forecast_cols].mean(axis=1)
result_df["minensemble"] = result_df[forecast_cols].min(axis=1)
result_df["maxensemble"] = result_df[forecast_cols].max(axis=1)
result_df["medianensemble"] = result_df[forecast_cols].median(axis=1)

ensembles=['meanensemble','medianensemble','maxensemble','minensemble']
result_df=result_df[['unique_id','ds']+ensembles]

result_df.to_csv('resultados_ensembles_mult.csv',index=False)

display(result_df)
print(result_df.describe())



,unique_id,ds,meanensemble,medianensemble,maxensemble,minensemble
0,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-10-05 06:00:00,1.167246,1.101322,1.414714,1.037918
1,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-10-05 12:00:00,1.677968,1.251858,3.018856,1.121058
2,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-10-05 18:00:00,1.263224,1.124228,1.808775,1.068434
3,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-10-06 00:00:00,1.185411,1.124127,1.500300,1.047652
4,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-10-06 06:00:00,1.157615,1.109027,1.398285,1.037918
...,...,...,...,...,...,...
4995,fc7ef67f-5553-4692-aebe-16da7cfe224b-b7d32e46-...,2025-10-29 06:00:00,0.854775,0.867188,0.964857,0.728998
4996,fc7ef67f-5553-4692-aebe-16da7cfe224b-b7d32e46-...,2025-10-29 12:00:00,0.998133,1.000000,1.105180,0.935886
4997,fc7ef67f-5553-4692-aebe-16da7cfe224b-b7d32e46-...,2025-10-29 18:00:00,1.044791,0.999019,1.287514,0.943083
4998,fc7ef67f-5553-4692-aebe-16da7cfe224b-b7d32e46-...,2025-10-30 00:00:00,0.822134,0.870058,0.964852,0.508271


                        ds  meanensemble  medianensemble  maxensemble  \
count                 5000   5000.000000     5000.000000  5000.000000   
mean   2025-10-18 00:07:12      0.923727        0.877045     1.296664   
min    2025-10-04 00:00:00      0.064618        0.062919     0.105195   
25%    2025-10-11 18:00:00      0.700791        0.655713     0.983899   
50%    2025-10-18 00:00:00      0.994545        0.984652     1.168881   
75%    2025-10-24 06:00:00      1.116508        1.089265     1.326402   
max    2025-10-30 18:00:00      2.872902        2.265625     7.021645   
std                    NaN      0.419408        0.347968     1.060392   

       minensemble  
count  5000.000000  
mean      0.702666  
min       0.000000  
25%       0.463456  
50%       0.805370  
75%       0.994492  
max       1.420933  
std       0.360522  


In [19]:
all_together=result_df[['unique_id','ds']+ensembles]\
    .merge(holdout_df,on=['unique_id','ds'])
all_together

,unique_id,ds,meanensemble,medianensemble,maxensemble,minensemble,y
0,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-10-05 06:00:00,1.167246,1.101322,1.414714,1.037918,1.121316
1,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-10-05 12:00:00,1.677968,1.251858,3.018856,1.121058,2.684848
2,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-10-05 18:00:00,1.263224,1.124228,1.808775,1.068434,1.097666
3,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-10-06 00:00:00,1.185411,1.124127,1.500300,1.047652,1.187510
4,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,2025-10-06 06:00:00,1.157615,1.109027,1.398285,1.037918,1.155408
...,...,...,...,...,...,...,...
4995,fc7ef67f-5553-4692-aebe-16da7cfe224b-b7d32e46-...,2025-10-29 06:00:00,0.854775,0.867188,0.964857,0.728998,0.702256
4996,fc7ef67f-5553-4692-aebe-16da7cfe224b-b7d32e46-...,2025-10-29 12:00:00,0.998133,1.000000,1.105180,0.935886,0.988115
4997,fc7ef67f-5553-4692-aebe-16da7cfe224b-b7d32e46-...,2025-10-29 18:00:00,1.044791,0.999019,1.287514,0.943083,1.387765
4998,fc7ef67f-5553-4692-aebe-16da7cfe224b-b7d32e46-...,2025-10-30 00:00:00,0.822134,0.870058,0.964852,0.508271,0.850331


In [ ]:
def rmse(a, f):
    return np.sqrt(np.mean((a - f) ** 2))

def mase(a, f):
    # a = actuals, f = forecasts
    scale = np.mean(np.abs(a[1:] - a[:-1]))  # naive lag-1 MAE
    return np.mean(np.abs(a - f)) / scale if scale != 0 else np.nan

metrics = []

# Metric grouping to see how these have performed 
for col in ensembles:
    m = (
        all_together.sort_values(["unique_id", "ds"])
        .groupby("unique_id")
        .apply(lambda g: pd.Series({
            "RMSE": rmse(g["y"].values, g[col].values),
            "MASE": mase(g["y"].values, g[col].values),
            "n": len(g)
        }))
        .reset_index()
    )
    m["model"] = col
    metrics.append(m)

metrics = pd.concat(metrics, ignore_index=True)

display(metrics)
metrics.groupby(["model"]).describe()

,unique_id,RMSE,MASE,n,model
0,093fde72-23c8-43d7-b24a-31ed1ed4e134-f774038a-...,0.481852,0.629781,100.0,meanensemble
1,0b6d5a76-9634-4313-bc93-9d320346b35e-71a24abd-...,0.374885,0.807074,100.0,meanensemble
2,1ac94e88-c453-45c8-a30f-e21c0e8077c5-622564d0-...,1.328435,0.625313,100.0,meanensemble
3,2463c639-c724-4caf-b922-13fa0b270442-71a24abd-...,0.624366,0.557935,100.0,meanensemble
4,269a1d98-de8d-4605-9bea-4953a1c1c815-f774038a-...,0.151776,0.514548,100.0,meanensemble
...,...,...,...,...,...
195,f774038a-5ffe-4a93-8baa-193e438ac942-093fde72-...,0.275116,0.886626,100.0,minensemble
196,f774038a-5ffe-4a93-8baa-193e438ac942-269a1d98-...,0.325728,1.008474,100.0,minensemble
197,f774038a-5ffe-4a93-8baa-193e438ac942-c856c5cb-...,0.199429,0.879316,100.0,minensemble
198,fc7ef67f-5553-4692-aebe-16da7cfe224b-9820891a-...,0.320380,0.779303,100.0,minensemble


RMSE                                                    \
               count      mean       std       min       25%       50%   
model                                                                    
maxensemble     50.0  0.932981  1.483929  0.106294  0.266376  0.413347   
meanensemble    50.0  0.713624  1.284220  0.122503  0.218089  0.333555   
medianensemble  50.0  0.702095  1.283219  0.120460  0.203341  0.329310   
minensemble     50.0  0.776411  1.286060  0.126998  0.244574  0.419929   

                                    MASE            ...                       \
                     75%       max count      mean  ...       75%        max   
model                                               ...                        
maxensemble     0.658107  7.644563  50.0  1.480122  ...  1.057966  14.953882   
meanensemble    0.606034  7.669272  50.0  0.686208  ...  0.670560   2.800075   
medianensemble  0.577612  7.664276  50.0  0.613470  ...  0.634883   0.988762   
minensemble     0.658277  7.716963  50.0  0.856704  ...  0.811998   6.678537   

                   n                                                 
               count   mean  std    min    25%    50%    75%    max  
model                                                                
maxensemble     50.0  100.0  0.0  100.0  100.0  100.0  100.0  100.0  
meanensemble    50.0  100.0  0.0  100.0  100.0  100.0  100.0  100.0  
medianensemble  50.0  100.0  0.0  100.0  100.0  100.0  100.0  100.0  
minensemble     50.0  100.0  0.0  100.0  100.0  100.0  100.0  100.0  

[4 rows x 24 columns]